# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research question: What recurring performance archetypes exist across FlyRank's content inventory, based on observable search and engagement signals?

The decision this supports: Given limited reviewer capacity, which broad strategy — protect, improve, rewrite, merge, prune, or monitor — should be applied to a page, before any individual human review begins. This is a triage-support question, not a prediction or diagnosis question.

Who acts on it: A content/SEO reviewer working through a large inventory who needs a starting map of "kinds of pages we have," rather than reading tens of thousands of rows individually.

Why this is framed as unsupervised, not supervised: No ground-truth label exists for "content archetype" — inventing one (e.g., borrowing FlyRank's own internal health_score or action_type) would risk building a model that just re-learns an existing product decision rather than discovering real structure in the data. This reasoning was established early (w02) and held throughout: every feature and every validation choice in this project was checked against it.

Cost of a wrong call: Lower-stakes than a binary decline/growth prediction, because a cluster assignment is a lens, not a verdict — no page loses traffic because it was mis-clustered. The real cost is wasted reviewer time and eroded trust in the system: if archetypes are named or interpreted carelessly, reviewers stop trusting the labels, and pages that genuinely need attention get deprioritized.

In [1]:
research_question = {
    "question": "What recurring performance archetypes exist across the content inventory, "
                 "based on observable search and engagement signals?",
    "decision_supported": "Triage priority (protect/improve/rewrite/merge/prune/monitor) "
                           "before individual human review",
    "task_type": "Unsupervised clustering — no label exists or was invented",
    "cost_of_wrong_call": "Low-to-moderate — wasted reviewer time and reduced trust in the "
                           "system, not a page-level harm, since output is decision-support only",
}
print(research_question["question"])
print("\nDecision supported:", research_question["decision_supported"])
print("Task type:", research_question["task_type"])

What recurring performance archetypes exist across the content inventory, based on observable search and engagement signals?

Decision supported: Triage priority (protect/improve/rewrite/merge/prune/monitor) before individual human review
Task type: Unsupervised clustering — no label exists or was invented


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Source: FlyRank internship warehouse (Hugging Face, FlyRank/internship-warehouse), queried live via DuckDB against Parquet files.

Tables used: fact_content_daily_performance (partition month=2026-03) joined to dim_content, both on client_hash_id + content_hash_id.

Date window: March 1–31, 2026 — a single, frozen mid-panel month. Deliberately not fact_content_daily_performance_sample, which is exactly June 2026, the panel's final month and the natural outcome window for any past→future label; using it during model development would risk contaminating a sealed test period.

What was excluded, and why — public-safe:

*  Any product-decision field (health_score, priority_score, action_type, recommendation, cluster, archetype) — excluded to prevent the model from re-learning an existing business rule instead of discovering real structure.

* GA4-sourced engagement fields — technically available, but severely sparse (4.21% of rows) and excluded from the core feature set for that reason (kept only as an optional post-cluster diagnostic).

* Rows outside GSC tracking coverage — the modeling population is necessarily the ~34% of the full inventory with real March search-visibility data; content with no GSC coverage that month is invisible to this analysis by construction, not by choice.

*  All client names, real URLs, and search query text — only hashed IDs and aggregate metrics are used or reported anywhere in this project.

In [2]:
import os, duckdb, numpy as np, pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# --- Window + grain verification ---
window_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT report_date) AS distinct_days
    FROM read_parquet('{TABLE}')
""").df()
print("Window verification:")
print(window_check.to_string(index=False))

grain_check = con.sql(f"""
    SELECT COUNT(*) AS duplicate_groups FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM read_parquet('{TABLE}') GROUP BY 1,2,3 HAVING COUNT(*) > 1
    )
""").df()
print("\nDuplicate grain groups (should be 0):", grain_check['duplicate_groups'].iloc[0])

# --- Availability rates ---
avail = con.sql(f"""
    SELECT COUNT(*) AS total,
           SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_avail,
           SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS ga4_avail
    FROM read_parquet('{TABLE}')
""").df().iloc[0]
gsc_pct = round(avail['gsc_avail'] / avail['total'] * 100, 2)
ga4_pct = round(avail['ga4_avail'] / avail['total'] * 100, 2)
print(f"\nGSC availability: {gsc_pct}%  |  GA4 availability: {ga4_pct}%")

# --- Full inventory vs. modeling population ---
dim_total = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{DIM}')").df().iloc[0]['n']

modeling_pop = con.sql(f"""
    SELECT COUNT(DISTINCT (client_hash_id, content_hash_id)) AS n
    FROM read_parquet('{TABLE}') WHERE gsc_data_available = TRUE
""").df().iloc[0]['n']

coverage_pct = round(modeling_pop / dim_total * 100, 2)
print(f"\nFull inventory (dim_content): {dim_total:,}")
print(f"Modeling population (GSC-available, this window): {modeling_pop:,}")
print(f"Coverage: {coverage_pct}% of full inventory")

data_summary = {
    "source": "FlyRank internship warehouse (Hugging Face)",
    "tables": ["fact_content_daily_performance (month=2026-03)", "dim_content"],
    "window": f"{window_check['min_date'].iloc[0]} to {window_check['max_date'].iloc[0]}",
    "raw_rows": int(window_check['total_rows'].iloc[0]),
    "distinct_days": int(window_check['distinct_days'].iloc[0]),
    "duplicate_grain_groups": int(grain_check['duplicate_groups'].iloc[0]),
    "gsc_availability_pct": gsc_pct,
    "ga4_availability_pct": ga4_pct,
    "full_inventory": int(dim_total),
    "modeling_population": int(modeling_pop),
    "coverage_pct_of_inventory": coverage_pct,
}
print("\n", data_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Window verification:
 total_rows   min_date   max_date  distinct_days
    9841378 2026-03-01 2026-03-31             31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Duplicate grain groups (should be 0): 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


GSC availability: 36.69%  |  GA4 availability: 4.21%

Full inventory (dim_content): 519,606
Modeling population (GSC-available, this window): 176,738
Coverage: 34.01% of full inventory

 {'source': 'FlyRank internship warehouse (Hugging Face)', 'tables': ['fact_content_daily_performance (month=2026-03)', 'dim_content'], 'window': '2026-03-01 00:00:00 to 2026-03-31 00:00:00', 'raw_rows': 9841378, 'distinct_days': 31, 'duplicate_grain_groups': 0, 'gsc_availability_pct': np.float64(36.69), 'ga4_availability_pct': np.float64(4.21), 'full_inventory': 519606, 'modeling_population': 176738, 'coverage_pct_of_inventory': np.float64(34.01)}


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Features (5 conceptual, 7 in the final model matrix): gsc_impressions, gsc_avg_position, gsc_clicks, content_age_days, word_count, plus two missingness/placeholder flags derived from the first two.

Preprocessing:



* gsc_avg_position = 0 (0.81% of rows) means no rank data, not a real position — converted to missing, imputed with the median of real values, and flagged (avg_position_missing_or_zero).

*  gsc_impressions and gsc_clicks are heavily right-skewed — log-transformed (log1p) before scaling to prevent a few extreme pages from dominating distance calculations.

* word_count was missing for 31.30% of rows — imputed with the population median and flagged (word_count_missing), so the model can distinguish "short content" from "unknown content length" rather than conflating them.

* All 7 features scaled with StandardScaler, fit on the training split only, then applied to validation — no leakage through the preprocessing step itself.


Label definition: None. This is unsupervised clustering by design (Section 1) — no proxy label was invented at any stage.

Baseline: A transparent, hand-coded rule (TITLE_META_CTR_FIX) — flag a page if it has ≥100 impressions, a usable (non-zero) position, and CTR at least 30% below the pooled expected CTR for its position tier. Built on two signals checked first: staleness (OPPOSITE — a real negative result, excluded) and CTR-vs-position (CONFIRMED — became the rule's basis).

Validation design: Client-grouped split (GroupShuffleSplit, 75/25) — pages from the same client plausibly share a CMS template, editorial convention, and tracking setup, so a naive random split risks letting a client's "house style" leak between train and validation.

Leakage checks: (1) static check confirming no product-decision fields ever entered the feature set; (2) a deliberate leak-trap — a cluster-derived proxy feature appended to the honest feature set inflates silhouette artificially, then removed, keeping only the honest number.


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

# --- Build modeling population ---
raw = con.sql(f"""
    WITH scoped AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{TABLE}') WHERE gsc_data_available = TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions, SUM(gsc_clicks) AS gsc_clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position
        FROM scoped GROUP BY client_hash_id, content_hash_id
    )
    SELECT a.*, d.content_created_date, d.word_count
    FROM agg a LEFT JOIN read_parquet('{DIM}') d
      ON a.client_hash_id = d.client_hash_id AND a.content_hash_id = d.content_hash_id
    ORDER BY a.client_hash_id, a.content_hash_id
""").df()

df = raw.copy()
df['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(df['content_created_date'])).dt.days
df['avg_position_missing_or_zero'] = (df['gsc_avg_position'] == 0).astype(int)
df['avg_position_clean'] = df['gsc_avg_position'].replace(0, np.nan)
avg_pos_median = df.loc[df['avg_position_clean'].notna(), 'avg_position_clean'].median()
df['avg_position_clean'] = df['avg_position_clean'].fillna(avg_pos_median)
df['log_gsc_impressions'] = np.log1p(df['gsc_impressions'])
df['log_gsc_clicks'] = np.log1p(df['gsc_clicks'])
df['word_count_missing'] = df['word_count'].isna().astype(int)
word_count_median = df['word_count'].median()
df['word_count'] = df['word_count'].fillna(word_count_median)

MODEL_FEATURES = ['log_gsc_impressions', 'avg_position_clean', 'log_gsc_clicks',
                   'content_age_days', 'word_count', 'avg_position_missing_or_zero', 'word_count_missing']

# --- Leakage check ---
forbidden_fields = {"health_score", "priority_score", "action_type", "recommended_action",
                     "recommendation", "cluster", "archetype", "label", "target"}
print("Forbidden fields in feature set:", forbidden_fields.intersection(set(MODEL_FEATURES)) or "NONE — clean")

# --- Client-grouped split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df['client_hash_id'].values))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[MODEL_FEATURES])
X_val = scaler.transform(val_df[MODEL_FEATURES])

print(f"\nTrain: {len(train_df)} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Val: {len(val_df)} rows, {val_df['client_hash_id'].nunique()} clients")
print("Client overlap:", len(set(train_df['client_hash_id']) & set(val_df['client_hash_id'])))

# --- Leak-trap demonstration ---
from sklearn.preprocessing import OneHotEncoder
km_honest = KMeans(n_clusters=4, random_state=42, n_init=30)
labels_honest = km_honest.fit_predict(X_train)
score_honest = silhouette_score(X_train, labels_honest, sample_size=20000, random_state=42)

leak_onehot = OneHotEncoder(sparse_output=False).fit_transform(labels_honest.reshape(-1, 1))
X_leaky = np.hstack([X_train, leak_onehot])
score_leaky = silhouette_score(X_leaky, labels_honest, sample_size=20000, random_state=42)

print(f"\nHonest silhouette: {score_honest:.4f}")
print(f"Leaky silhouette (cluster-derived proxy appended): {score_leaky:.4f}")
print(f"Jump: +{score_leaky - score_honest:.4f} — leak removed, honest number kept")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Forbidden fields in feature set: NONE — clean

Train: 133474 rows, 35 clients
Val: 43264 rows, 12 clients
Client overlap: 0

Honest silhouette: 0.2944
Leaky silhouette (cluster-derived proxy appended): 0.3541
Jump: +0.0597 — leak removed, honest number kept


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Model vs. baseline, same population, honest split, real numbers:

# Model Validation Summary

## 1. Week 4 baseline

- **Population:** eligible rows (impressions≥100, position>0)
- **Output:** `TITLE_META_CTR_FIX` queue
- **Metric:** coverage
- **Result:** 61,267 flagged

---

## 2. K-Means (k=4)

- **Population:** 176,738 rows, client-grouped split
- **Output:** 4 clusters
- **Metric:** validation silhouette
- **Result:** 0.3568

---

## 3. Baseline overlay

- **Population:** same rows
- **Output:** lift by cluster
- **Metric:** max/min lift
- **Result:** 1.197 / 0.000

---

## 4. Stability check

- **Population:** same rows
- **Output:** seed consistency (5 seeds)
- **Metric:** mean ARI
- **Result:** ≥0.996

---

## 5. Split honesty check

- **Population:** same model, same data
- **Output:** naive vs. grouped split
- **Metric:** silhouette gap
- **Result:** 0.2962 (naive, 45/47 client overlap) vs. 0.3568 (grouped, 0 overlap)

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

# --- 1. k-selection sweep ---
k_rows = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=30)
    tl = km.fit_predict(X_train)
    vl = km.predict(X_val)
    k_rows.append({
        'k': k,
        'val_silhouette': round(silhouette_score(X_val, vl, sample_size=20000, random_state=42), 4),
        'val_davies_bouldin': round(davies_bouldin_score(X_val, vl), 4),
    })
k_df = pd.DataFrame(k_rows)
print("K-selection sweep:")
print(k_df.to_string(index=False))

# --- 2. Final model, k=4 ---
FINAL_K = 4
final_km = KMeans(n_clusters=FINAL_K, random_state=42, n_init=30)
train_labels = final_km.fit_predict(X_train)
val_labels = final_km.predict(X_val)
train_df['cluster'] = train_labels
val_df['cluster'] = val_labels
full_labeled = pd.concat([train_df.assign(split='train'), val_df.assign(split='val')], ignore_index=True)

print(f"\nSelected k=4 | val silhouette: {k_df[k_df.k==4]['val_silhouette'].iloc[0]} "
      f"| val Davies-Bouldin: {k_df[k_df.k==4]['val_davies_bouldin'].iloc[0]}")

# --- 3. Seed stability check (5 seeds) ---
seeds = [7, 13, 29, 42, 101]
seed_labels = {s: KMeans(n_clusters=FINAL_K, random_state=s, n_init=30).fit_predict(X_train) for s in seeds}
aris = [adjusted_rand_score(seed_labels[seeds[i]], seed_labels[seeds[j]])
        for i in range(len(seeds)) for j in range(i+1, len(seeds))]
print(f"\nSeed stability — mean ARI: {np.mean(aris):.4f} | min ARI: {min(aris):.4f}")

# --- 4. Naive vs. grouped split comparison (before/after honesty check) ---
train_naive, val_naive = train_test_split(df, test_size=0.25, random_state=42)
scaler_n = StandardScaler()
Xn_train = scaler_n.fit_transform(train_naive[MODEL_FEATURES])
Xn_val = scaler_n.transform(val_naive[MODEL_FEATURES])
km_naive = KMeans(n_clusters=4, random_state=42, n_init=30)
km_naive.fit(Xn_train)
naive_val_labels = km_naive.predict(Xn_val)
naive_sil = silhouette_score(Xn_val, naive_val_labels, sample_size=20000, random_state=42)
naive_overlap = len(set(train_naive['client_hash_id']) & set(val_naive['client_hash_id']))
print(f"\nNaive split: val silhouette={naive_sil:.4f}, client overlap={naive_overlap}/{df['client_hash_id'].nunique()}")
print(f"Grouped split: val silhouette={k_df[k_df.k==4]['val_silhouette'].iloc[0]}, client overlap=0")

# --- 5. Cluster profiles, for archetype identity confirmation (index numbers are arbitrary per fit) ---
print("\nCluster profiles (median values, for archetype identity mapping):")
cluster_profiles = full_labeled.groupby('cluster')[['gsc_impressions', 'gsc_clicks', 'gsc_avg_position']].median()
print(cluster_profiles.to_string())

# --- 6. Baseline rebuild, on the CURRENT full_labeled, in the same cell as its own use ---
def position_tier(pos):
    if pos <= 3: return 'pos_1_3'
    elif pos <= 10: return 'pos_4_10'
    elif pos <= 20: return 'pos_11_20'
    else: return 'pos_21_plus'

baseline_base = full_labeled[(full_labeled['gsc_impressions'] >= 100) & (full_labeled['gsc_avg_position'] > 0)].copy()
baseline_base['ctr'] = baseline_base['gsc_clicks'] / baseline_base['gsc_impressions']
baseline_base['position_tier'] = baseline_base['gsc_avg_position'].apply(position_tier)
tier_ctr = baseline_base.groupby('position_tier').apply(
    lambda g: g['gsc_clicks'].sum() / g['gsc_impressions'].sum(), include_groups=False
).to_dict()
baseline_base['expected_ctr'] = baseline_base['position_tier'].map(tier_ctr)
baseline_base['ctr_gap_pct'] = (baseline_base['expected_ctr'] - baseline_base['ctr']).clip(lower=0) / baseline_base['expected_ctr']
baseline_base['baseline_flag'] = (baseline_base['ctr_gap_pct'] >= 0.30).astype(int)

print(f"\nBaseline queue: {baseline_base['baseline_flag'].sum()} rows flagged (of {len(baseline_base)} eligible)")
print("\nPosition-tier expected CTR:")
for tier, ctr in tier_ctr.items():
    print(f"  {tier}: {ctr:.4%}")

# --- 7. Baseline lift by cluster — the final comparison table ---
global_rate = baseline_base['baseline_flag'].mean()
lift_final = baseline_base.groupby('cluster').agg(
    rows=('cluster', 'size'), flagged=('baseline_flag', 'sum')
).reset_index()
lift_final['flag_rate'] = (lift_final['flagged'] / lift_final['rows']).round(4)
lift_final['lift'] = (lift_final['flag_rate'] / global_rate).round(3)

# Cluster 3 (no-rank-data) is structurally absent from baseline_base — add it explicitly, lift=0
if 3 not in lift_final['cluster'].values:
    lift_final = pd.concat([lift_final, pd.DataFrame([{
        'cluster': 3, 'rows': int((full_labeled['cluster'] == 3).sum()),
        'flagged': 0, 'flag_rate': 0.0, 'lift': 0.0
    }])], ignore_index=True).sort_values('cluster').reset_index(drop=True)

print("\nBaseline lift by cluster (final, verified):")
print(lift_final.to_string(index=False))

results_summary = {
    "k_selected": FINAL_K,
    "val_silhouette": float(k_df[k_df.k==4]['val_silhouette'].iloc[0]),
    "val_davies_bouldin": float(k_df[k_df.k==4]['val_davies_bouldin'].iloc[0]),
    "seed_stability_mean_ari": round(float(np.mean(aris)), 4),
    "naive_split_silhouette": round(float(naive_sil), 4),
    "naive_split_client_overlap": naive_overlap,
    "grouped_split_client_overlap": 0,
    "baseline_queue_rows": int(baseline_base['baseline_flag'].sum()),
    "baseline_lift_by_cluster": lift_final.to_dict(orient='records'),
}
print("\n", results_summary)

K-selection sweep:
 k  val_silhouette  val_davies_bouldin
 3          0.3485              1.1340
 4          0.3568              0.9087
 5          0.3416              1.1002
 6          0.3395              0.9820
 7          0.3314              1.0163
 8          0.3270              0.9931

Selected k=4 | val silhouette: 0.3568 | val Davies-Bouldin: 0.9087

Seed stability — mean ARI: 0.9981 | min ARI: 0.9963

Naive split: val silhouette=0.2962, client overlap=45/47
Grouped split: val silhouette=0.3568, client overlap=0

Cluster profiles (median values, for archetype identity mapping):
         gsc_impressions  gsc_clicks  gsc_avg_position
cluster                                               
0                   96.0         0.0         15.232108
1                 2533.0         6.0          6.613053
2                   38.0         0.0          7.500000
3                    1.0         0.0          0.000000

Baseline queue: 61267 rows flagged (of 101441 eligible)

Position-tier expec

## 5. Limitations

*What this work cannot claim.*

Coverage is partial, not universal. The modeling population (176,738 content items) represents 34.01% of the full inventory (519,606 items in dim_content) — verified fresh this session. Content outside GSC's tracked coverage that month is invisible to this analysis entirely, not by choice but by construction. Any archetype described here describes the tracked subset of the inventory, not the whole catalog.

GA4/engagement signal is nearly absent. Only 4.21% of rows had GA4 data available this session — confirmed identical to every prior verification. The clusters therefore describe search-side visibility almost exclusively; they say very little about on-page engagement or reading behavior for the vast majority of content.

Single-month snapshot — no trend or seasonality claim possible. March 2026 is one month out of the roughly 17 available in the full panel. Nothing here shows whether a page's archetype is stable over time, whether these patterns are seasonal, or whether a page would move between archetypes in a different month. A different month could show different clusters entirely.

Two of four archetypes are partly defined by data availability, not pure behavior. The Missing/Imputed Metadata cluster is substantially characterized by an elevated word_count_missing flag rather than genuine content-depth variation — per-cluster example-row inspection during this project found sampled rows carrying the identical population-median imputed value, not real per-page word counts. The Sparse/No-Rank Risk cluster is, by definition, the population with no real ranking data. Membership in either archetype can reflect an instrumentation or tracking gap as much as it reflects actual content behavior.

Client-population generalization is validated; per-client accuracy is not guaranteed. The client-grouped validation split (0.3568 silhouette, 0 client overlap, confirmed fresh this session) shows the clustering generalizes to unseen clients as a population. It does not guarantee accuracy for any specific client — particularly small clients, or any client not represented in the 47 seen this session.

Split design measurably changes the reported number. A naive random split (45/47 clients overlapping train/val) produced a materially different validation silhouette (0.2962) than the honest grouped split (0.3568) on identical data and an identical model. This is itself a limitation worth stating: the reported metric is sensitive to methodological choices, and a differently-designed validation could report a different number from the same underlying model.

The baseline-cluster relationship required correction, and may still hold surprises. During final verification for this paper, the baseline-lift-by-cluster figures were found to differ meaningfully from an earlier draft's numbers, traced to a computation issue not initially obvious from aggregate checks alone. This was caught and corrected here — but it's a concrete demonstration that even a well-tested pipeline can produce numbers that look plausible while being wrong, and that a single verification pass is not automatically sufficient.

No causal claims anywhere. Every finding in this project — the baseline signal verdicts, the cluster-vs-baseline lift, feature ablation, the split-design comparison — is observed and directional within this dataset, never causal. A confirmed pattern means the pattern held in this specific data; it does not mean acting on it will produce any particular outcome. Cluster names are descriptive labels for review triage, never quality judgments or guarantees.

In [5]:
limitations = {
    "coverage_pct_of_full_inventory": 34.01,
    "ga4_availability_pct": 4.21,
    "temporal_scope": "single month (March 2026), no trend/seasonality claim possible",
    "missingness_driven_clusters": ["Missing/Imputed Metadata", "Sparse/No-Rank Risk"],
    "validation_scope": "client-population generalization validated; per-client accuracy not guaranteed",
    "split_sensitivity": {
        "naive_split_silhouette": 0.2962,
        "grouped_split_silhouette": 0.3568,
        "note": "same data, same model, different split design -> different reported metric"
    },
    "correction_note": "baseline-lift-by-cluster figures were found and corrected during "
                        "final verification for this notebook — see Section 4",
    "claim_standard": "all findings are observed/directional; no causal claims anywhere in this work",
}
import json
print(json.dumps(limitations, indent=2))

{
  "coverage_pct_of_full_inventory": 34.01,
  "ga4_availability_pct": 4.21,
  "temporal_scope": "single month (March 2026), no trend/seasonality claim possible",
  "missingness_driven_clusters": [
    "Missing/Imputed Metadata",
    "Sparse/No-Rank Risk"
  ],
  "validation_scope": "client-population generalization validated; per-client accuracy not guaranteed",
  "split_sensitivity": {
    "naive_split_silhouette": 0.2962,
    "grouped_split_silhouette": 0.3568,
    "note": "same data, same model, different split design -> different reported metric"
  },
  "correction_note": "baseline-lift-by-cluster figures were found and corrected during final verification for this notebook \u2014 see Section 4",
  "claim_standard": "all findings are observed/directional; no causal claims anywhere in this work"
}


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

This section reuses w07's playbook logic — CTR-fix baseline flag plus cluster archetype context plus confidence tags — but with the corrected lift values from Section 4. The ranking priority changes as a result of the correction: archetypes are no longer ordered by the originally-reported (incorrect) lift, but by the verified numbers.

Priority order, by verified baseline lift:

1.Missing/Imputed Metadata (lift 1.296) — highest concentration of CTR-fix flags, but confidence must be discounted: this archetype is partly defined by missing word_count data (Section 5), so a flag here may reflect a tracking gap as much as a genuine CTR problem. Recommend: data-quality check before CTR action.

2. Low-Traffic/Emerging (lift 1.275) — second-highest concentration, genuinely low-traffic pages with real CTR gaps. Recommend: standard CTR-fix review, but expect many flagged rows to have thin click volume (per w07's LOW_CLICK_VOLUME_CAUTION logic), which makes individual CTR estimates noisy.

3. Core Established Content (lift 0.690) — below the global average, the opposite of what an earlier draft of this paper reported. High-traffic pages are, almost by definition, less likely to show a large CTR gap. Recommend: use archetype as context, not a trigger — the CTR-fix heuristic is less concentrated here, so a flag on one of these pages is comparatively more informative when it does occur.

4. Sparse/No-Rank Risk (lift 0.000) — structurally invisible to the baseline rule (its position > 0 filter excludes this cluster by construction). Recommend: prioritize data-quality/indexing investigation over CTR optimization — absence of a flag here is not evidence of health.

Tiebreaker note: a large share of Missing/Imputed Metadata rows tie at the maximum possible CTR gap (ctr_gap_pct = 1.0, i.e., zero clicks against a non-zero expected CTR). Sorting by impressions as a secondary key surfaces the pages with the most traffic at stake among these ties — but it also surfaces the same client-concentration pattern first identified in an earlier phase of this work: client_73cda7b4e4f265ea alone accounts for 6 of the top 10 rows, and only 5 unique clients appear across the top 20 (7 across the top 50). Every one of these top rows already carries a DATA_QUALITY_CAUTION tag (Missing/Imputed archetype) and, for the repeat client, a CLIENT_CONCENTRATION_CAUTION tag as well.

This is the correct behavior of the confidence system, not a flaw in it: the highest-scoring rows in this queue are exactly the ones the caution tags say to trust least. A reviewer should treat the top of this list as "investigate the pattern," not "apply N individual fixes" — the recurrence of one client across two independent phases of this project suggests a client-level tracking or GSC-integration issue is more likely than 6+ independent content problems.

In [6]:
CLUSTER_ARCHETYPES = {
    0: "Missing/Imputed Metadata",
    1: "Core Established Content",
    2: "Low-Traffic/Emerging",
    3: "Sparse/No-Rank Risk",
}
# Verified lift, this session — used to set priority order, NOT to gate which pages get flagged
ARCHETYPE_LIFT = {0: 1.296, 1: 0.690, 2: 1.275, 3: 0.000}
ARCHETYPE_PRIORITY_RANK = {0: 1, 2: 2, 1: 3, 3: 4}  # by descending lift

ACTION_LABEL = "TITLE_META_CTR_FIX"
REASON_CODE = "HIGH_VISIBILITY_LOW_CTR_VS_POSITION"
LOW_CLICK_THRESHOLD = 10
CLIENT_CONCENTRATION_THRESHOLD = 0.30
DATA_QUALITY_RISK_CLUSTERS = {0, 3}  # Missing/Imputed Metadata, Sparse/No-Rank

queue = baseline_base[baseline_base['baseline_flag'] == 1].copy()
queue['archetype'] = queue['cluster'].map(CLUSTER_ARCHETYPES)
queue['archetype_lift'] = queue['cluster'].map(ARCHETYPE_LIFT)
queue['action_label'] = ACTION_LABEL
queue['reason_code'] = REASON_CODE

client_share_per_cluster = full_labeled.groupby('cluster')['client_hash_id'].apply(
    lambda x: x.value_counts(normalize=True)
)

def get_confidence_tags(row):
    tags = []
    if row['cluster'] in DATA_QUALITY_RISK_CLUSTERS:
        tags.append("DATA_QUALITY_CAUTION")
    if row['gsc_clicks'] < LOW_CLICK_THRESHOLD:
        tags.append("LOW_CLICK_VOLUME_CAUTION")
    share = client_share_per_cluster.get((row['cluster'], row['client_hash_id']), 0)
    if share >= CLIENT_CONCENTRATION_THRESHOLD:
        tags.append("CLIENT_CONCENTRATION_CAUTION")
    return tags if tags else ["HIGH_CONFIDENCE"]

queue['confidence_tags'] = queue.apply(get_confidence_tags, axis=1)

# Rank: archetype priority (by verified lift) first, then score within archetype
queue['archetype_priority'] = queue['cluster'].map(ARCHETYPE_PRIORITY_RANK)
queue = queue.sort_values(['archetype_priority', 'ctr_gap_pct'], ascending=[True, False]).reset_index(drop=True)
queue['rank'] = queue.index + 1

output_cols = ['rank', 'client_hash_id', 'content_hash_id', 'action_label', 'reason_code',
               'archetype', 'archetype_lift', 'confidence_tags', 'gsc_impressions',
               'gsc_clicks', 'gsc_avg_position', 'ctr_gap_pct']

print("Total ranked actions:", len(queue))
print("\nConfidence tag distribution:")
print(queue['confidence_tags'].apply(lambda x: x[0] if len(x)==1 else 'MULTIPLE_CAUTIONS').value_counts())
print("\nArchetype distribution within the queue:")
print(queue['archetype'].value_counts())
print("\nTop 10 ranked actions:")
print(queue[output_cols].head(10).to_string(index=False))

# Add impressions as a tiebreaker among rows with identical ctr_gap_pct —
# among equally "bad" CTR gaps, prioritize the one with more real traffic at stake
queue = queue.sort_values(
    ['archetype_priority', 'ctr_gap_pct', 'gsc_impressions'],
    ascending=[True, False, False]
).reset_index(drop=True)
queue['rank'] = queue.index + 1

print("Top 10 ranked actions (with impressions as tiebreaker):")
print(queue[output_cols].head(10).to_string(index=False))

# Also worth showing explicitly: how many distinct clients appear in the top 20
print("\nUnique clients in top 20:", queue.head(20)['client_hash_id'].nunique())
print("Unique clients in top 50:", queue.head(50)['client_hash_id'].nunique())

Total ranked actions: 61267

Confidence tag distribution:
confidence_tags
LOW_CLICK_VOLUME_CAUTION    37802
MULTIPLE_CAUTIONS           20233
HIGH_CONFIDENCE              3231
DATA_QUALITY_CAUTION            1
Name: count, dtype: int64

Archetype distribution within the queue:
archetype
Low-Traffic/Emerging        20809
Missing/Imputed Metadata    20234
Core Established Content    20224
Name: count, dtype: int64

Top 10 ranked actions:
 rank          client_hash_id          content_hash_id       action_label                         reason_code                archetype  archetype_lift                                  confidence_tags  gsc_impressions  gsc_clicks  gsc_avg_position  ctr_gap_pct
    1 client_08a6a72ff48e62c0 content_000f4b73532b9e9d TITLE_META_CTR_FIX HIGH_VISIBILITY_LOW_CTR_VS_POSITION Missing/Imputed Metadata           1.296 [DATA_QUALITY_CAUTION, LOW_CLICK_VOLUME_CAUTION]            100.0         0.0         15.940000          1.0
    2 client_08a6a72ff48e62c0 content_00

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

This section generates the final, corrected tables and figures that the deployed HTML paper needs — replacing the incorrect lift values currently live on the page, and producing a real chart image where the deployed version currently uses CSS-only bars. All outputs saved to work/figures/ (committed) and work/outputs/ (JSON receipts committed; queue CSV not committed, per the export contract).

In [7]:
import matplotlib.pyplot as plt
import json, os

os.makedirs('work/figures', exist_ok=True)
os.makedirs('work/outputs', exist_ok=True)

# --- Artifact 1: k-selection sweep table (for Methodology section) ---
k_df.to_csv('work/outputs/capstone_k_selection.csv', index=False)
print("Saved: work/outputs/capstone_k_selection.csv")

# --- Artifact 2: Cluster profiles table (for Results section) ---
cluster_profile_export = full_labeled.groupby('cluster').agg(
    rows=('cluster', 'size'),
    clients=('client_hash_id', 'nunique'),
    median_impressions=('gsc_impressions', 'median'),
    median_clicks=('gsc_clicks', 'median'),
    median_position=('gsc_avg_position', 'median'),
).reset_index()
cluster_profile_export['archetype'] = cluster_profile_export['cluster'].map(CLUSTER_ARCHETYPES)
cluster_profile_export.to_csv('work/outputs/capstone_cluster_profiles.csv', index=False)
print("Saved: work/outputs/capstone_cluster_profiles.csv")
print(cluster_profile_export.to_string(index=False))

# --- Artifact 3: CORRECTED baseline lift by cluster (replaces the wrong deployed numbers) ---
lift_export = lift_final.copy()
lift_export['archetype'] = lift_export['cluster'].map(CLUSTER_ARCHETYPES)
lift_export.to_csv('work/outputs/capstone_baseline_lift_by_cluster.csv', index=False)
print("\nSaved: work/outputs/capstone_baseline_lift_by_cluster.csv")
print(lift_export.to_string(index=False))

# --- Figure 1: Baseline lift by archetype — REAL chart, replacing the deployed CSS bars ---
fig, ax = plt.subplots(figsize=(9, 5))
plot_order = lift_export.sort_values('lift', ascending=True)
colors = ['#c0392b' if l < 1.0 else '#245b86' for l in plot_order['lift']]
ax.barh(plot_order['archetype'], plot_order['lift'], color=colors)
ax.axvline(x=1.0, color='gray', linestyle='--', linewidth=1, label='Global average (lift = 1.0)')
ax.set_xlabel('Lift (flag rate ÷ global flag rate)')
ax.set_title('Baseline CTR-Fix Flag Lift by Content Archetype\n(corrected — see Section 5 for correction note)')
ax.legend()
plt.tight_layout()
plt.savefig('work/figures/capstone_baseline_lift_by_cluster.png', dpi=150)
plt.close()
print("\nSaved: work/figures/capstone_baseline_lift_by_cluster.png")

# --- Figure 2: k-selection sweep (validation silhouette vs k) ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_df['k'], k_df['val_silhouette'], marker='o', color='#245b86', linewidth=2)
ax.scatter([4], k_df[k_df.k==4]['val_silhouette'], color='#c0392b', s=120, zorder=5, label='Selected k=4')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('Validation silhouette score')
ax.set_title('K-Selection Sweep — Validation Silhouette by k')
ax.legend()
plt.tight_layout()
plt.savefig('work/figures/capstone_k_selection_sweep.png', dpi=150)
plt.close()
print("Saved: work/figures/capstone_k_selection_sweep.png")

# --- Figure 3: Naive vs grouped split comparison ---
fig, ax = plt.subplots(figsize=(7, 5))
splits = ['Naive random\n(45/47 client overlap)', 'Client-grouped\n(0 client overlap)']
sils = [0.2962, 0.3568]
ax.bar(splits, sils, color=['#c0392b', '#245b86'])
ax.set_ylabel('Validation silhouette score')
ax.set_title('Split Design Comparison — Before/After Client Grouping')
for i, v in enumerate(sils):
    ax.text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('work/figures/capstone_split_comparison.png', dpi=150)
plt.close()
print("Saved: work/figures/capstone_split_comparison.png")

# --- Final consolidated JSON receipt for the whole paper ---
paper_artifacts = {
    "data": {
        "raw_rows": 9841378, "window": "2026-03-01 to 2026-03-31",
        "modeling_population": 176738, "coverage_pct": 34.01,
        "gsc_availability_pct": 36.69, "ga4_availability_pct": 4.21,
    },
    "methodology": {
        "leak_trap_honest_silhouette": 0.2944, "leak_trap_leaky_silhouette": 0.3541,
        "forbidden_fields_found": "none",
    },
    "results": {
        "k_selected": 4, "val_silhouette": 0.3568, "val_davies_bouldin": 0.9087,
        "seed_stability_mean_ari": 0.9981,
        "naive_split_silhouette": 0.2962, "naive_split_client_overlap": 45,
        "grouped_split_client_overlap": 0,
        "baseline_queue_rows": 61267,
        "baseline_lift_by_cluster": lift_export.to_dict(orient='records'),
    },
    "recommendations": {
        "total_ranked_actions": 61267,
        "confidence_tag_counts": {
            "LOW_CLICK_VOLUME_CAUTION": 37802, "MULTIPLE_CAUTIONS": 20233,
            "HIGH_CONFIDENCE": 3231, "DATA_QUALITY_CAUTION": 1,
        },
    },
    "correction_note": "baseline lift values were found incorrect in an earlier draft and "
                        "corrected during final capstone-notebook verification; see Section 5",
}
with open('work/outputs/capstone_paper_artifacts.json', 'w') as f:
    json.dump(paper_artifacts, f, indent=2)
print("\nSaved: work/outputs/capstone_paper_artifacts.json")

Saved: work/outputs/capstone_k_selection.csv
Saved: work/outputs/capstone_cluster_profiles.csv
 cluster  rows  clients  median_impressions  median_clicks  median_position                archetype
       0 52572       23                96.0            0.0        15.232108 Missing/Imputed Metadata
       1 48766       33              2533.0            6.0         6.613053 Core Established Content
       2 73966       47                38.0            0.0         7.500000     Low-Traffic/Emerging
       3  1434       30                 1.0            0.0         0.000000      Sparse/No-Rank Risk

Saved: work/outputs/capstone_baseline_lift_by_cluster.csv
 cluster  rows  flagged  flag_rate  lift                archetype
       0 25856    20234     0.7826 1.296 Missing/Imputed Metadata
       1 48555    20224     0.4165 0.690 Core Established Content
       2 27030    20809     0.7698 1.275     Low-Traffic/Emerging
       3  1434        0     0.0000 0.000      Sparse/No-Rank Risk

Saved: wor

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
